# 03a · External datasets — explore (DMID + CDD-CESM)

**Runtime: CPU is enough** (Runtime → Change runtime type → CPU). Does not touch the training runs.

1. Downloads DMID from figshare and the CDD-CESM annotation files from TCIA into Drive.
2. Prints what is inside (folders, file types, mask format, counts) and shows a few examples.

Send me the full output of cells 2 and 3; I will write the crop/test notebook from it.
CDD-CESM *images* need the IBM Aspera plugin, so they are not downloaded here.


In [ ]:
# 1 · Drive + folders
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, collections, subprocess, requests
EXT = '/content/drive/MyDrive/quantum-placement/data/external'
for d in ('dmid_raw', 'cdd_cesm'):
    os.makedirs(f'{EXT}/{d}', exist_ok=True)
print('external data ->', EXT)


In [ ]:
# 2 · DMID: download from figshare, extract, describe
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
print('Title:', art.get('title')); print('License:', (art.get('license') or {}).get('name'))
print('Description (first 1500 chars):\n', (art.get('description') or '')[:1500])
for f in art['files']:
    print(f"  file: {f['name']}  {f['size']/1e6:.1f} MB")

RAW = f'{EXT}/dmid_raw'
for f in art['files']:
    dst = f"{RAW}/{f['name']}"
    if not os.path.exists(dst):
        with requests.get(f['download_url'], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dst, 'wb') as out:
                for chunk in r.iter_content(1 << 20): out.write(chunk)
        print('downloaded', f['name'])

LOCAL = '/content/dmid'                      # extract on local disk (fast), not on Drive
os.makedirs(LOCAL, exist_ok=True)
for a in glob.glob(f'{RAW}/*.rar') + glob.glob(f'{RAW}/*.zip'):
    if a.endswith('.rar'):
        if subprocess.run(['which', 'unrar'], capture_output=True).returncode:
            subprocess.run(['apt-get', '-qq', 'install', '-y', 'unrar'], check=False)
        subprocess.run(['unrar', 'x', '-o+', '-inul', a, LOCAL + '/'], check=False)
    else:
        subprocess.run(['unzip', '-q', '-o', a, '-d', LOCAL], check=False)

print('\nFolder tree (files per folder, by extension):')
for root, dirs, files in sorted(os.walk(LOCAL)):
    if files:
        ext = collections.Counter(os.path.splitext(f)[1].lower() for f in files)
        print(f'  {root.replace(LOCAL, "") or "/"}: {len(files)} files {dict(ext)}  e.g. {sorted(files)[:3]}')

import pandas as pd
for t in glob.glob(f'{LOCAL}/**/*.csv', recursive=True) + glob.glob(f'{LOCAL}/**/*.xls*', recursive=True):
    df = pd.read_csv(t) if t.endswith('.csv') else pd.read_excel(t)
    print(f'\nTABLE {t.replace(LOCAL, "")}: shape {df.shape}'); print(df.head(8).to_string())
    for c in df.columns:
        if df[c].nunique() < 15: print(f'   {c}: {df[c].value_counts(dropna=False).to_dict()}')
for t in glob.glob(f'{LOCAL}/**/*.txt', recursive=True)[:3]:
    print(f'\nTEXT {t.replace(LOCAL, "")}:'); print(open(t, errors='ignore').read()[:1500])


In [ ]:
# 2b · DMID: look at masks (format, values) + 3 examples
import numpy as np, cv2, matplotlib.pyplot as plt
masks = [p for p in glob.glob(f'{LOCAL}/**/*', recursive=True)
         if os.path.isfile(p) and ('mask' in p.lower() or 'seg' in p.lower() or 'roi' in p.lower())
         and p.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'))]
print('candidate mask files:', len(masks)); print(masks[:5])
for p in masks[:5]:
    m = cv2.imread(p, cv2.IMREAD_UNCHANGED)
    if m is None: continue
    print(os.path.basename(p), m.shape, m.dtype, 'unique values:', np.unique(m)[:10], 'fg frac:', round(float((m > 0).mean()), 4))
fig, ax = plt.subplots(1, min(3, len(masks)), figsize=(12, 4), squeeze=False)
for a, p in zip(ax[0], masks[:3]):
    m = cv2.imread(p, 0); a.imshow(m, cmap='gray'); a.set_title(os.path.basename(p)[:30], fontsize=8); a.axis('off')
plt.show()


In [ ]:
# 3 · CDD-CESM: annotation files (direct download) + summary
BASE = 'https://www.cancerimagingarchive.net/wp-content/uploads/'
CDD = f'{EXT}/cdd_cesm'
for name in ('Radiology_hand_drawn_segmentations_v2.csv', 'Radiology-manual-annotations.xlsx'):
    if not os.path.exists(f'{CDD}/{name}'):
        r = requests.get(BASE + name, timeout=120); r.raise_for_status(); open(f'{CDD}/{name}', 'wb').write(r.content)
seg = pd.read_csv(f'{CDD}/Radiology_hand_drawn_segmentations_v2.csv')
seg['shape'] = seg['region_shape_attributes'].apply(lambda s: json.loads(s).get('name', 'none') if isinstance(s, str) and s.strip() else 'none')
seg['type'] = seg['#filename'].str.extract(r'_(DM|CM)_')[0]
print('segmentation rows:', len(seg), '| images:', seg['#filename'].nunique())
print(pd.crosstab(seg['type'], seg['shape']).to_string())

xl = pd.read_excel(f'{CDD}/Radiology-manual-annotations.xlsx', sheet_name=None)
for sh, df in xl.items():
    print(f'\nSHEET {sh}: shape {df.shape}'); print('columns:', list(df.columns)); print(df.head(5).to_string())
    for c in df.columns:
        if 1 < df[c].nunique() < 15: print(f'   {c}: {df[c].value_counts(dropna=False).to_dict()}')
